In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    to_timestamp,
    current_timestamp,
    max as spark_max,
    lit,
    count,
    when
)

from delta.tables import DeltaTable

In [0]:
bronze_table = "observatorio_dev.bronze.generacion_real"
silver_table = "observatorio_dev.silver.generacion_real"

print("Bronze table:", bronze_table)
print("Silver table:", silver_table)

In [0]:
bronze_full_df = spark.table(bronze_table)

bronze_full_df.printSchema()
display(bronze_full_df.limit(10))

### Obtener ultimo ingestion_timestamp cargado en silver

In [0]:
last_ingestion_timestamp = (
    spark.table(silver_table)
    .select(
        spark_max("ingestion_timestamp").alias("last_ingestion_timestamp")
    )
    .collect()[0]["last_ingestion_timestamp"]
)

print("Último ingestion_timestamp cargado en Silver:", last_ingestion_timestamp)

### Leer solo registros nuevos desde Bronze

In [0]:
bronze_df = spark.table(bronze_table)

if last_ingestion_timestamp is not None:
    bronze_df = bronze_df.filter(
        col("ingestion_timestamp") > lit(last_ingestion_timestamp)
    )

print("Registros nuevos desde Bronze:", bronze_df.count())

display(bronze_df.limit(20))

### Transformar Bronze a Silver

In [0]:
silver_df = (
    bronze_df
    .select(
        trim(upper(col("codigo_variable"))).alias("codigo_variable"),
        to_timestamp(col("fecha_hora")).alias("fecha_hora"),
        trim(upper(col("codigo_duracion"))).alias("codigo_duracion"),
        trim(upper(col("unidad_medida"))).alias("unidad_medida"),
        trim(upper(col("codigo_sic_agente"))).alias("codigo_agente"),
        trim(upper(col("codigo_planta"))).alias("codigo_planta"),
        trim(upper(col("version"))).alias("version"),
        col("valor").cast("double").alias("valor"),

        col("source_file_name"),
        col("source_file_path"),
        col("ingestion_timestamp"),
        col("load_date"),

        current_timestamp().alias("silver_created_at"),
        current_timestamp().alias("silver_updated_at")
    )
    .dropDuplicates([
        "fecha_hora",
        "codigo_agente",
        "codigo_planta",
        "codigo_variable",
        "codigo_duracion",
        "version"
    ])
)

print("Registros transformados para Silver:", silver_df.count())

display(silver_df.limit(20))

### Merge Incremental

In [0]:
new_records_count = silver_df.count()

print("Registros nuevos a procesar:", new_records_count)

if new_records_count == 0:
    print("No hay registros nuevos para cargar en Silver.")

else:
    target = DeltaTable.forName(spark, silver_table)

    (
        target.alias("target")
        .merge(
            silver_df.alias("source"),
            """
            target.fecha_hora = source.fecha_hora
            AND target.codigo_agente = source.codigo_agente
            AND target.codigo_planta = source.codigo_planta
            AND target.codigo_variable = source.codigo_variable
            AND target.codigo_duracion = source.codigo_duracion
            AND target.version = source.version
            """
        )
        .whenMatchedUpdate(set={
            "unidad_medida": "source.unidad_medida",
            "valor": "source.valor",
            "source_file_name": "source.source_file_name",
            "source_file_path": "source.source_file_path",
            "ingestion_timestamp": "source.ingestion_timestamp",
            "load_date": "source.load_date",
            "silver_updated_at": "source.silver_updated_at"
        })
        .whenNotMatchedInsert(values={
            "codigo_variable": "source.codigo_variable",
            "fecha_hora": "source.fecha_hora",
            "codigo_duracion": "source.codigo_duracion",
            "unidad_medida": "source.unidad_medida",
            "codigo_agente": "source.codigo_agente",
            "codigo_planta": "source.codigo_planta",
            "version": "source.version",
            "valor": "source.valor",
            "source_file_name": "source.source_file_name",
            "source_file_path": "source.source_file_path",
            "ingestion_timestamp": "source.ingestion_timestamp",
            "load_date": "source.load_date",
            "silver_created_at": "source.silver_created_at",
            "silver_updated_at": "source.silver_updated_at"
        })
        .execute()
    )

    print("MERGE ejecutado correctamente.")

### Validacion Final

In [0]:
%sql
SELECT * 
FROM observatorio_dev.silver.generacion_real
LIMIT 10;

In [0]:
%sql
SELECT
    COUNT(*) AS registros,
    MIN(fecha_hora) AS fecha_minima,
    MAX(fecha_hora) AS fecha_maxima,
    COUNT(DISTINCT version) AS versiones_distintas
FROM observatorio_dev.silver.generacion_real;

In [0]:
%sql
WITH duplicados AS (
    SELECT
        fecha_hora,
        codigo_agente,
        codigo_planta,
        codigo_variable,
        codigo_duracion,
        version,
        COUNT(*) AS cantidad
    FROM observatorio_dev.silver.generacion_real
    GROUP BY
        fecha_hora,
        codigo_agente,
        codigo_planta,
        codigo_variable,
        codigo_duracion,
        version
    HAVING COUNT(*) > 1
)

SELECT COUNT(*) AS grupos_duplicados
FROM duplicados;